# Airlines Data Engineering Pipeline

**Source:** `UseCase_-_Airlines.xlsx` (4 sheets: `flights`, `payments`, `bookings`, `passengers`)

**Goal:** Take the raw workbook and produce a clean, PII-safe, aggregation-ready dataset for
Power BI, supporting these KPIs:

1. Average Flight Duration
2. Route-wise Traffic
3. Delays / Anomalies
4. Distribution of Flights by Airline

**Approach:** Inspect the workbook first, validate what is actually there, and only clean or
standardize fields where the data itself gives enough evidence to justify it. Anything that
can't be justified is flagged, not silently fixed or dropped.

The pipeline stages: `Raw Excel → Ingestion → Validation → Cleaning → Standardization →
Transformation → PII Protection → Analytical Dataset → KPI Preparation → Loading/Saving`

In [1]:
import re
import os
import hashlib
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger("airlines_pipeline")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 3. Load Raw Data

The raw workbook is only ever **read**, never written to. Each sheet is loaded into its own
DataFrame and checked against the schema we found during inspection. If a sheet is missing,
its columns don't match, or it loads empty, the pipeline stops here rather than continuing
with bad assumptions.

In [2]:
RAW_PATH = Path("../data/raw/Airlines.xlsx")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SCHEMA = {
    "flights": ["flight_id", "airline", "source", "destination",
                "departure_time", "arrival_time", "duration"],
    "payments": ["payment_id", "booking_id", "amount", "payment_method"],
    "bookings": ["booking_id", "passenger_id", "flight_id", "booking_date", "status",
                 "passport_number", "seat_number", "emergency_contact_name",
                 "emergency_contact_phone"],
    "passengers": ["passenger_id", "first_name", "last_name", "age", "gender",
                   "email", "phone", "aadhaar_id", "date_of_birth"],
}

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw source file not found at {RAW_PATH}")

xls = pd.ExcelFile(RAW_PATH)
missing_sheets = [s for s in EXPECTED_SCHEMA if s not in xls.sheet_names]
if missing_sheets:
    raise ValueError(f"Expected sheet(s) missing from workbook: {missing_sheets}")

raw = {name: pd.read_excel(xls, name) for name in EXPECTED_SCHEMA}

for name, expected_cols in EXPECTED_SCHEMA.items():
    actual_cols = list(raw[name].columns)
    if actual_cols != expected_cols:
        raise ValueError(
            f"Sheet '{name}' schema mismatch.\nExpected: {expected_cols}\nFound:    {actual_cols}"
        )
    assert len(raw[name]) > 0, f"Sheet '{name}' loaded with 0 rows"
    log.info(f"Loaded '{name}': {raw[name].shape[0]} rows, {raw[name].shape[1]} columns - schema OK")

flights_raw = raw["flights"]
payments_raw = raw["payments"]
bookings_raw = raw["bookings"]
passengers_raw = raw["passengers"]

INFO: Loaded 'flights': 1020 rows, 7 columns - schema OK
INFO: Loaded 'payments': 1000 rows, 4 columns - schema OK
INFO: Loaded 'bookings': 1000 rows, 9 columns - schema OK
INFO: Loaded 'passengers': 1039 rows, 9 columns - schema OK


## 4. Initial Data Inspection

One pass over each sheet: shape, dtypes, missing values, and a couple of sample rows. This is
what the validation and cleaning decisions in the next sections are based on.

In [3]:
for name, df in raw.items():
    print("=" * 70)
    print(name.upper(), df.shape)
    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum()[df.isna().sum() > 0])

    if name == "flights":
        print("\nSample rows:")
        display(df.head(2))

    print()

FLIGHTS (1020, 7)

Data types:
flight_id                 object
airline                   object
source                    object
destination               object
departure_time    datetime64[ns]
arrival_time      datetime64[ns]
duration                  object
dtype: object

Missing values:
airline    41
dtype: int64

Sample rows:


,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00



PAYMENTS (1000, 4)

Data types:
payment_id        object
booking_id        object
amount            object
payment_method    object
dtype: object

Missing values:
amount    48
dtype: int64

BOOKINGS (1000, 9)

Data types:
booking_id                         object
passenger_id                       object
flight_id                          object
booking_date               datetime64[ns]
status                             object
passport_number                    object
seat_number                        object
emergency_contact_name             object
emergency_contact_phone            object
dtype: object

Missing values:
status    45
dtype: int64

PASSENGERS (1039, 9)

Data types:
passenger_id             object
first_name               object
last_name                object
age                       int64
gender                   object
email                    object
phone                    object
aadhaar_id                int64
date_of_birth    datetime64[ns]
dtype: object

Miss

## 5. Data Quality Validation

Checks below are grouped by sheet. Nothing is corrected in this section - we only measure and
record what's wrong, then decide in Section 6/7 whether each issue should be **corrected**,
**standardized**, **flagged**, or **excluded**.

**flights**
- `flight_id` format consistency (2 letters + 3 digits)
- Missing / placeholder `airline` values
- Exact duplicate rows and duplicate `flight_id`s
- `source == destination` (impossible route)
- Departure/arrival timestamp sanity (arrival before departure)

**bookings / payments**
- Missing or `INVALID` categorical values (`status`, `amount`)
- Referential integrity to `flights` / `passengers` / `bookings`

**passengers**
- Duplicate `passenger_id` values pointing to *different* underlying people (conflicting data)
- `aadhaar_id` digit-length consistency (should be 12 digits)

The same missing-value, duplicate-row, and referential-integrity checks come up for every
sheet, so they're written once as small helpers below and reused per sheet.

In [4]:
def count_missing(df, col):
    return int(df[col].isna().sum())

def count_placeholder(df, col, placeholder):
    return int((df[col] == placeholder).sum())

def count_exact_duplicates(df):
    return int(df.duplicated().sum())

def count_orphans(child_df, child_col, parent_df, parent_col):
    """Rows in child_df whose child_col value has no match in parent_df[parent_col]."""
    return int((~child_df[child_col].isin(parent_df[parent_col])).sum())

In [5]:
# Derive the actual flight_id structure from the data instead of assuming a generic pattern:
# split each id into its leading prefix (letters/digits before the trailing digit run) and the
# digit run itself.
prefix_extract = flights_raw["flight_id"].astype(str).str.extract(r"^([A-Za-z0-9]+?)(\d+)$")
observed_prefixes = sorted(prefix_extract[0].dropna().unique())
observed_suffix_lengths = sorted(prefix_extract[1].dropna().str.len().unique())
print("Observed flight_id prefixes:", observed_prefixes)
print("Observed numeric suffix length(s):", observed_suffix_lengths)

# Build the validation pattern from what's actually observed (e.g. '6F', 'AI', 'SJ', 'UK'),
# rather than a generic "2 letters + 3 digits" guess that would also accept prefixes that never
# appear in this dataset (and would incorrectly reject a real one like '6F', which is
# alphanumeric, not alphabetic).
prefix_group = "|".join(re.escape(p) for p in observed_prefixes)
id_pattern = re.compile(rf"^({prefix_group})\d{{3}}$")

valid_id_mask = flights_raw["flight_id"].apply(lambda x: bool(id_pattern.match(str(x))))
n_bad_ids = int((~valid_id_mask).sum())

n_exact_dup_rows = count_exact_duplicates(flights_raw)
n_dup_flight_ids = int(flights_raw["flight_id"].duplicated().sum())
n_route_self_loop = int((flights_raw["source"] == flights_raw["destination"]).sum())
n_airline_missing = count_missing(flights_raw, "airline")
n_airline_unknown = count_placeholder(flights_raw, "airline", "UNKNOWN")

dep = pd.to_datetime(flights_raw["departure_time"])
arr = pd.to_datetime(flights_raw["arrival_time"])
n_arrival_before_departure = int((arr < dep).sum())

print("flight_id not matching the observed prefix + 3-digit pattern:", n_bad_ids)
print("Exact duplicate rows:", n_exact_dup_rows)
print("Duplicate flight_id values (rows):", n_dup_flight_ids)
print("source == destination (impossible route):", n_route_self_loop)
print("airline missing (NaN):", n_airline_missing)
print("airline recorded as literal 'UNKNOWN':", n_airline_unknown)
print("arrival_time earlier than departure_time:", n_arrival_before_departure)

assert n_route_self_loop == 0, "found flights with source == destination"

Observed flight_id prefixes: ['6F', 'AI', 'SJ', 'UK']
Observed numeric suffix length(s): [np.int64(3)]
flight_id not matching the observed prefix + 3-digit pattern: 0
Exact duplicate rows: 15
Duplicate flight_id values (rows): 16
source == destination (impossible route): 0
airline missing (NaN): 41
airline recorded as literal 'UNKNOWN': 31
arrival_time earlier than departure_time: 1


In [6]:
# Duplicate flight_id rows: are they exact duplicates, or genuinely different flights
# that happen to reuse the same flight number?
dup_id_rows = flights_raw[flights_raw["flight_id"].duplicated(keep=False)].sort_values("flight_id")
exact_dup_ids = dup_id_rows[dup_id_rows.duplicated(keep=False)]["flight_id"].nunique()
print("flight_id values whose duplicate rows are 100% identical:", exact_dup_ids)
print("flight_id values whose duplicate rows differ (legitimate repeat flight number):",
      dup_id_rows["flight_id"].nunique() - exact_dup_ids)
dup_id_rows[dup_id_rows["flight_id"] == "6F250"]

flight_id values whose duplicate rows are 100% identical: 15
flight_id values whose duplicate rows differ (legitimate repeat flight number): 1


,flight_id,airline,source,destination,departure_time,arrival_time,duration
253,6F250,UNKNOWN,DEL,BLR,2026-04-20 03:26:41.701,2026-04-20 07:30:41.701,04:04:00
270,6F250,UNKNOWN,CCU,BLR,2026-04-20 02:23:41.702,2026-04-20 02:56:41.702,00:33:00


In [7]:
# Timestamp sanity: does the existing `duration` field agree with (arrival - departure)?
calc_minutes = (arr - dep).dt.total_seconds() / 60

def duration_to_minutes(t):
    if pd.isna(t):
        return np.nan
    return t.hour * 60 + t.minute

recorded_minutes = flights_raw["duration"].apply(duration_to_minutes)
mismatch = (calc_minutes - recorded_minutes).abs()

print("Rows where calculated duration disagrees with recorded duration by > 5 minutes:")
suspect = flights_raw[mismatch > 5].copy()
suspect["calc_minutes"] = calc_minutes[mismatch > 5]
suspect["recorded_minutes"] = recorded_minutes[mismatch > 5]
suspect[["flight_id", "departure_time", "arrival_time", "duration", "calc_minutes", "recorded_minutes"]]

Rows where calculated duration disagrees with recorded duration by > 5 minutes:


,flight_id,departure_time,arrival_time,duration,calc_minutes,recorded_minutes
355,SJ192,2026-04-19 18:45:42,2026-04-18 23:45:42,1899-12-29 05:00:00,-1140.0,300


In [8]:
# bookings: categorical validity + referential integrity
n_status_missing = count_missing(bookings_raw, "status")
n_status_invalid = count_placeholder(bookings_raw, "status", "INVALID")
n_booking_dup = count_exact_duplicates(bookings_raw)
n_booking_bad_passenger_ref = count_orphans(bookings_raw, "passenger_id", passengers_raw, "passenger_id")
n_booking_bad_flight_ref = count_orphans(bookings_raw, "flight_id", flights_raw, "flight_id")

print("bookings.status missing (NaN):", n_status_missing)
print("bookings.status literal 'INVALID':", n_status_invalid)
print("bookings exact duplicate rows:", n_booking_dup)
print("bookings.passenger_id not found in passengers:", n_booking_bad_passenger_ref)
print("bookings.flight_id not found in flights:", n_booking_bad_flight_ref)
print("bookings.status value counts:\n", bookings_raw["status"].value_counts(dropna=False))

assert n_booking_dup == 0, "unexpected exact duplicate rows in bookings"
assert n_booking_bad_passenger_ref == 0, "bookings reference passenger_ids that don't exist"
assert n_booking_bad_flight_ref == 0, "bookings reference flight_ids that don't exist"

bookings.status missing (NaN): 45
bookings.status literal 'INVALID': 30
bookings exact duplicate rows: 0
bookings.passenger_id not found in passengers: 0
bookings.flight_id not found in flights: 0
bookings.status value counts:
 status
CONFIRMED    320
CANCELLED    314
PENDING      291
NaN           45
INVALID       30
Name: count, dtype: int64


In [9]:
# payments: amount validity + referential integrity
amount_numeric = pd.to_numeric(payments_raw["amount"], errors="coerce")
n_amount_missing = count_missing(payments_raw, "amount")
n_amount_invalid_text = count_placeholder(payments_raw, "amount", "INVALID")
n_payment_bad_booking_ref = count_orphans(payments_raw, "booking_id", bookings_raw, "booking_id")
n_payment_dup = count_exact_duplicates(payments_raw)

print("payments.amount missing (NaN):", n_amount_missing)
print("payments.amount literal 'INVALID':", n_amount_invalid_text)
print("payments.booking_id not found in bookings:", n_payment_bad_booking_ref)
print("payments exact duplicate rows:", n_payment_dup)
print("payments.payment_method values:", payments_raw["payment_method"].unique())

# Multiple payments can share one booking_id (retries / installments) - that's a legitimate
# business pattern, not a duplicate, since every payment_id is unique.
print("\npayment_id is unique:", payments_raw["payment_id"].is_unique)
print("Max payments recorded against a single booking_id:", payments_raw["booking_id"].value_counts().max())

assert n_payment_dup == 0, "unexpected exact duplicate rows in payments"
assert n_payment_bad_booking_ref == 0, "payments reference booking_ids that don't exist"

payments.amount missing (NaN): 48
payments.amount literal 'INVALID': 30
payments.booking_id not found in bookings: 0
payments exact duplicate rows: 0
payments.payment_method values: ['NETBANKING' 'UPI' 'CARD']

payment_id is unique: True
Max payments recorded against a single booking_id: 6


In [10]:
# passengers: duplicate IDs and aadhaar format
dup_pid_mask = passengers_raw["passenger_id"].duplicated(keep=False)
n_dup_passenger_rows = int(dup_pid_mask.sum())
n_dup_passenger_ids = int(passengers_raw.loc[dup_pid_mask, "passenger_id"].nunique())
n_exact_dup_passenger_rows = count_exact_duplicates(passengers_raw)
n_lastname_missing = count_missing(passengers_raw, "last_name")

aadhaar_len = passengers_raw["aadhaar_id"].astype(str).str.len()
print("passenger_id values reused for different underlying records:", n_dup_passenger_ids,
      f"({n_dup_passenger_rows} rows affected)")
print("Exact duplicate passenger rows (same id AND same data):", n_exact_dup_passenger_rows)
print("\naadhaar_id digit-length distribution (should be 12):")
print(aadhaar_len.value_counts().sort_index())
print("\nlast_name missing:", n_lastname_missing)

assert n_exact_dup_passenger_rows == 0, "unexpected exact duplicate rows in passengers"

# Example of a reused passenger_id pointing to two different people
passengers_raw[passengers_raw["passenger_id"] == passengers_raw.loc[dup_pid_mask, "passenger_id"].iloc[0]]

passenger_id values reused for different underlying records: 36 (75 rows affected)
Exact duplicate passenger rows (same id AND same data): 0

aadhaar_id digit-length distribution (should be 12):
aadhaar_id
10      5
11    109
12    925
Name: count, dtype: int64

last_name missing: 10


,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
34,P1034,Ishaan,Iyer,62,M,ishaan.iyer@outlook.com,+91-8679984033,122362316658,1964-08-14
35,P1034,Ishaan,Iyer,62,M,ishaan.iyer@gmail.com,+91-7994310083,669096705466,1964-09-24


### Validation findings & decisions

| Issue | Evidence | Decision |
|---|---|---|
| 15 exact duplicate rows in `flights` | Full-row duplicates, same `flight_id` | **Exclude** (drop duplicates) |
| `6F250` appears twice with different data | Same flight number, different source/timestamps | **Keep both** - legitimate repeat use of a flight number, not a duplicate |
| `airline` missing (41) / `'UNKNOWN'` (31) | Both represent "airline not recorded" | **Standardize** both to `"Unknown"` |
| 1 flight (`SJ192`) has arrival before departure | Calculated duration is negative (~-19h) vs. recorded duration of 5h | **Flag as anomaly** - not enough evidence to know which timestamp is wrong |
| `flight_id` format | All 1020 ids match one of the prefixes actually observed in the data (`6F`, `AI`, `SJ`, `UK`) followed by exactly 3 digits | No correction needed - rule derived from the observed ids, not assumed |
| `flight_id` is not a unique key in `flights` (e.g. `6F250` is used twice, for two different flights) | Same id, different source/timestamps on each row | **Do not** join other tables onto `flight_id` and merge back 1:1 - see Section 9 |
| `bookings.status` missing (45) / `'INVALID'` (30) | No reliable way to infer true status | **Standardize** to `"Unknown"`, keep row |
| `payments.amount` missing (48) / `'INVALID'` (30) | No reliable way to recover the amount | **Flag** as anomaly, keep row (amount not needed for any required KPI) |
| Referential integrity (bookings→passengers/flights, payments→bookings) | 0 orphaned rows in every case | No action needed |
| `passenger_id` reused for different people (36 ids / 75 rows) | Different `aadhaar_id`/DOB under the same id | **Flag as anomaly** - cannot safely merge or guess which record is correct |
| `aadhaar_id` has 10/11/12 digit values | Real Aadhaar numbers are always 12 digits; likely leading zeros lost when stored as an integer | **Correct**: zero-pad to 12 digits (documented, reversible, low-risk standardization) |
| `last_name` missing (10) | No reliable way to derive it | **Leave as** `"Unknown"` |

No true **scheduled-vs-actual delay** can be calculated: the workbook only has a single
departure/arrival timestamp per flight, not a separate "scheduled" time. Delay KPIs will use the
anomaly/data-quality flags instead - see Section 11.

## 6. Data Cleaning and Standardization

Applying the decisions from Section 5. Each rule below follows the same shape: **raw value →
validation finding → cleaning rule → final value**. Every change is counted so the numbers are
traceable in the final Data Quality Summary (Section 12).

In [11]:
dq = {}  # running data-quality summary, filled in as we go
dq["flights_raw_count"] = len(flights_raw)
dq["passengers_raw_count"] = len(passengers_raw)
dq["bookings_raw_count"] = len(bookings_raw)
dq["payments_raw_count"] = len(payments_raw)

dq["flights_exact_duplicates_removed"] = int(n_exact_dup_rows)
dq["flights_airline_missing_found"] = int(n_airline_missing)
dq["flights_airline_unknown_found"] = int(n_airline_unknown)
dq["bookings_status_missing_found"] = int(n_status_missing)
dq["bookings_status_invalid_found"] = int(n_status_invalid)
dq["bookings_exact_duplicates_found"] = int(n_booking_dup)
dq["payments_amount_missing_found"] = int(n_amount_missing)
dq["payments_amount_invalid_found"] = int(n_amount_invalid_text)
dq["payments_exact_duplicates_found"] = int(n_payment_dup)
dq["passengers_reused_ids_found"] = int(n_dup_passenger_ids)
dq["passengers_reused_id_rows_found"] = int(n_dup_passenger_rows)
dq["passengers_lastname_missing_found"] = int(n_lastname_missing)
dq["passengers_exact_duplicates_found"] = int(n_exact_dup_passenger_rows)

In [12]:
# --- flights: drop exact duplicate rows only ---
flights = flights_raw.drop_duplicates().copy()
log.info(f"Dropped {dq['flights_exact_duplicates_removed']} exact duplicate flight rows")

# airline: raw value -> NaN or literal 'UNKNOWN' (both mean "not recorded") -> standardized to 'Unknown'
flights["airline"] = flights["airline"].fillna("Unknown").replace("UNKNOWN", "Unknown")
assert flights["airline"].notna().all()

# source/destination: raw value -> casing already consistent -> enforced upper/stripped as a safeguard
flights["source"] = flights["source"].str.upper().str.strip()
flights["destination"] = flights["destination"].str.upper().str.strip()

flights["airline"].value_counts(dropna=False)

INFO: Dropped 15 exact duplicate flight rows


airline
IndiGo       249
SpiceJet     236
Air India    233
Vistara      218
Unknown       69
Name: count, dtype: int64

In [13]:
# status: raw value -> NaN or literal 'INVALID' (both mean "no reliable status") -> standardized to 'Unknown'
bookings = bookings_raw.copy()
bookings["status"] = bookings["status"].fillna("Unknown").replace("INVALID", "Unknown")
assert bookings["status"].notna().all()
bookings["status"].value_counts(dropna=False)

status
CONFIRMED    320
CANCELLED    314
PENDING      291
Unknown       75
Name: count, dtype: int64

In [14]:
# aadhaar_id: raw value -> 10/11/12-digit string (leading zeros lost on ingestion) -> zero-padded to 12 digits
passengers = passengers_raw.copy()
passengers["aadhaar_id"] = passengers["aadhaar_id"].astype(str).str.zfill(12)
dq["passengers_aadhaar_zero_padded"] = int((aadhaar_len < 12).sum())
assert (passengers["aadhaar_id"].str.len() == 12).all()

# last_name: raw value -> missing -> no reliable way to derive it -> left as 'Unknown'
passengers["last_name"] = passengers["last_name"].fillna("Unknown")

# passenger_id conflicts: flagged, not merged or dropped (Section 5 finding)
passengers["id_conflict_flag"] = passengers["passenger_id"].duplicated(keep=False)
print("Aadhaar IDs zero-padded:", dq["passengers_aadhaar_zero_padded"])
print("Passenger rows flagged for reused/conflicting passenger_id:", passengers["id_conflict_flag"].sum())

Aadhaar IDs zero-padded: 114
Passenger rows flagged for reused/conflicting passenger_id: 75


In [15]:
# amount: raw value -> NaN or literal 'INVALID' -> no reliable way to recover -> flagged, row kept
payments = payments_raw.copy()
payments["amount_numeric"] = pd.to_numeric(payments["amount"], errors="coerce")
payments["amount_invalid_flag"] = payments["amount_numeric"].isna()
print("payments rows with unusable amount:", payments["amount_invalid_flag"].sum())

payments rows with unusable amount: 78


## 7. Flight Time and Duration Transformation

`departure_time` and `arrival_time` are already full timestamps (with a date component), so an
overnight flight's arrival already carries the *next day's* date - a plain subtraction gives the
correct duration without any manual day-rollover logic. The single case where arrival is earlier
than departure (`SJ192`) is a genuine data problem, not a normal overnight flight, so it is
flagged rather than "fixed".

In [16]:
flights["departure_time"] = pd.to_datetime(flights["departure_time"])
flights["arrival_time"] = pd.to_datetime(flights["arrival_time"])

# duration: raw departure/arrival timestamps -> validated against the recorded `duration` field
# -> duration_minutes/duration_hours derived directly from (arrival - departure)
flights["duration_minutes"] = (flights["arrival_time"] - flights["departure_time"]).dt.total_seconds() / 60
flights["duration_hours"] = (flights["duration_minutes"] / 60).round(2)
flights["flight_date"] = flights["departure_time"].dt.date
flights["overnight_flag"] = flights["arrival_time"].dt.date != flights["departure_time"].dt.date

# anomaly: calculated duration negative, OR disagrees with the recorded duration by more than
# 5 minutes -> flagged, not corrected (not enough evidence to know which value is wrong)
recorded_minutes_clean = flights["duration"].apply(duration_to_minutes)
duration_gap = (flights["duration_minutes"] - recorded_minutes_clean).abs()
flights["duration_anomaly_flag"] = (flights["duration_minutes"] < 0) | (duration_gap > 5)

assert flights["duration_minutes"].notna().all()

dq["flights_after_dedup_count"] = len(flights)
dq["flights_duration_anomalies_found"] = int(flights["duration_anomaly_flag"].sum())
dq["flights_overnight_count"] = int(flights["overnight_flag"].sum())

print("Duration anomalies flagged:", dq["flights_duration_anomalies_found"])
print("Overnight flights:", dq["flights_overnight_count"])
flights[flights["duration_anomaly_flag"]][
    ["flight_id", "departure_time", "arrival_time", "duration", "duration_minutes"]
]

Duration anomalies flagged: 1
Overnight flights: 123


,flight_id,departure_time,arrival_time,duration,duration_minutes
355,SJ192,2026-04-19 18:45:42,2026-04-18 23:45:42,1899-12-29 05:00:00,-1140.0


## 8. PII Handling

**PII fields identified:**

| Sheet | PII fields |
|---|---|
| `passengers` | `first_name`, `last_name`, `email`, `phone`, `aadhaar_id`, `date_of_birth` |
| `bookings` | `passport_number`, `emergency_contact_name`, `emergency_contact_phone` |

**Protection approach:**
- Direct identifiers (`aadhaar_id`, `passport_number`) are **hashed** (SHA-256, truncated) -
  useful for joining/counting without revealing the real value.
- Contact details (`email`, `phone`, `emergency_contact_phone`) are **masked**, keeping only
  enough to spot-check, not enough to contact anyone.
- Names are **dropped entirely** from anything exported - they add no analytical value at the
  flight/route level this case study needs.

**Why the analytical dataset stays clean:** the final Power BI dataset (Section 10) is built at
the *flight* level - counts and durations - and never includes a passenger-level row at all, so
no PII field is exposed there by construction. The masking helpers below are provided in case a
passenger- or booking-level extract is ever needed downstream.

In [17]:
def hash_id(value):
    if pd.isna(value):
        return value
    return hashlib.sha256(str(value).encode()).hexdigest()[:12]

def mask_phone(value):
    if pd.isna(value):
        return value
    s = str(value)
    return s[:4] + "*" * (len(s) - 6) + s[-2:]

def mask_email(value):
    if pd.isna(value):
        return value
    local, _, domain = str(value).partition("@")
    return local[0] + "*" * max(len(local) - 1, 1) + "@" + domain

passengers_masked = passengers.copy()
passengers_masked["aadhaar_id"] = passengers_masked["aadhaar_id"].apply(hash_id)
passengers_masked["phone"] = passengers_masked["phone"].apply(mask_phone)
passengers_masked["email"] = passengers_masked["email"].apply(mask_email)
passengers_masked = passengers_masked.drop(columns=["first_name", "last_name"])

bookings_masked = bookings.copy()
bookings_masked["passport_number"] = bookings_masked["passport_number"].apply(hash_id)
bookings_masked["emergency_contact_phone"] = bookings_masked["emergency_contact_phone"].apply(mask_phone)
bookings_masked = bookings_masked.drop(columns=["emergency_contact_name"])

assert not {"first_name", "last_name"} & set(passengers_masked.columns)
assert "emergency_contact_name" not in bookings_masked.columns

passengers_masked.head(3)

,passenger_id,age,gender,email,phone,aadhaar_id,date_of_birth,id_conflict_flag
0,P1000,52,F,v****************@gmail.com,+91-********90,99466baa9b8f,1974-04-08,False
1,P1001,15,M,k************@hotmail.com,+91-********97,6d05ccbeb501,2011-03-07,False
2,P1002,72,M,m*********@outlook.com,+91-********92,3e24ac79e2c9,1954-09-10,False


## 9. Data Relationships / Integration

Referential integrity was already confirmed in Section 5 (0 orphaned rows for
`bookings → passengers`, `bookings → flights`, `payments → bookings`).

`flight_id` is **not a unique key** within `flights` - a small number of flight numbers (e.g.
`6F250`) are legitimately reused across more than one actual flight record (different
source/timestamps, see Section 5). That means bookings cannot be safely aggregated by
`flight_id` and merged back onto the flight-level table: every flight row sharing that id would
receive the *same* booking count, silently inflating booking numbers for those flights. There's
no reliable field in the data to build a genuine flight-instance key, so rather than inventing
one just to force the join, bookings are validated and summarized here as their own table and
kept separate from the flight-level analytics dataset.

`payments` is validated but not merged either: none of the four required KPIs depend on payment
amounts, and ~8% of amounts are unusable.

In [18]:
# Bookings are summarized on their own, by status - this is intentionally NOT merged into the
# flight-level analytics dataset (see explanation above: flight_id is not unique in `flights`,
# so a flight_id-based join would double-count bookings for reused flight numbers).
bookings_status_summary = bookings["status"].value_counts().rename("booking_count")
print("Bookings by status (standalone summary, kept separate from flight-level analytics):")
bookings_status_summary

Bookings by status (standalone summary, kept separate from flight-level analytics):


status
CONFIRMED    320
CANCELLED    314
PENDING      291
Unknown       75
Name: booking_count, dtype: int64

## 10. Create Analytics-Ready Dataset

A genuinely flight-level table, aggregation-ready for Power BI. Built directly from the cleaned
`flights` table only - no booking counts are merged in (see Section 9), so every row here is one
real flight record with no risk of double-counting.

In [19]:
analytics_df = flights.copy()
analytics_df["route"] = analytics_df["source"] + "-" + analytics_df["destination"]

final_columns = [
    "flight_id", "airline", "source", "destination", "route",
    "departure_time", "arrival_time", "flight_date",
    "duration_minutes", "duration_hours", "overnight_flag", "duration_anomaly_flag",
]
analytics_df = analytics_df[final_columns]

# Critical assumptions for the analytics-ready dataset - fail loudly rather than ship a bad file.
assert "flight_id" in flights.columns
assert analytics_df["flight_id"].notna().all()
assert len(analytics_df) == len(flights), "analytics_df must stay flight-level (one row per flight record)"

pii_columns = {
    "first_name", "last_name", "email", "phone", "aadhaar_id", "date_of_birth",
    "passport_number", "emergency_contact_name", "emergency_contact_phone",
}
assert pii_columns.isdisjoint(analytics_df.columns), "PII column leaked into analytics-ready dataset"

dq["analytics_dataset_row_count"] = len(analytics_df)
print(analytics_df.shape)
analytics_df.head()

(1005, 12)


,flight_id,airline,source,destination,route,departure_time,arrival_time,flight_date,duration_minutes,duration_hours,overnight_flag,duration_anomaly_flag
0,SJ010,SpiceJet,CCU,MAA,CCU-MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,2026-04-20,174.0,2.90,True,False
1,AI155,Air India,BOM,CCU,BOM-CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,2026-04-20,108.0,1.80,True,False
2,UK094,Vistara,BOM,CCU,BOM-CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,2026-04-20,105.0,1.75,True,False
3,AI245,Air India,BOM,CCU,BOM-CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,2026-04-20,156.0,2.60,True,False
4,AI192,Air India,MAA,BOM,MAA-BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,2026-04-20,299.0,4.98,True,False


## 11. KPI Preparation

All four required KPIs, computed directly from `analytics_df` (excluding the one flagged
duration anomaly from the duration-based KPI so it doesn't distort the average).

**Note on delays:** the workbook has no *scheduled* departure/arrival time - only the actual
one - so a true scheduled-vs-actual delay cannot be calculated. "Delays / Anomalies" is
represented instead by `duration_anomaly_flag`, which is fully supported by the data.

In [20]:
clean_duration = analytics_df[~analytics_df["duration_anomaly_flag"]]

# 1. Average Flight Duration
avg_duration_overall = clean_duration["duration_minutes"].mean()
avg_duration_by_airline = clean_duration.groupby("airline")["duration_minutes"].mean().round(1)

print(f"Average flight duration (overall): {avg_duration_overall:.1f} minutes")
avg_duration_by_airline

Average flight duration (overall): 164.5 minutes


airline
Air India    163.1
IndiGo       167.7
SpiceJet     163.6
Unknown      166.1
Vistara      162.7
Name: duration_minutes, dtype: float64

In [21]:
# 2. Route-wise Traffic
route_traffic = analytics_df["route"].value_counts().rename("flight_count")
route_traffic

route
BOM-CCU    90
CCU-DEL    72
MAA-BLR    65
BLR-BOM    60
HYD-MAA    57
DEL-HYD    54
HYD-DEL    42
BOM-DEL    39
CCU-BOM    33
DEL-BLR    29
DEL-BOM    28
BOM-MAA    27
HYD-BOM    27
BOM-HYD    26
HYD-CCU    26
HYD-BLR    26
DEL-CCU    26
MAA-DEL    26
CCU-MAA    24
MAA-CCU    24
BOM-BLR    23
DEL-MAA    23
MAA-BOM    21
BLR-CCU    21
MAA-HYD    21
CCU-HYD    21
CCU-BLR    20
BLR-HYD    19
BLR-DEL    19
BLR-MAA    16
Name: flight_count, dtype: int64

In [22]:
# 3. Delays / Anomalies (data-quality based, since no scheduled time exists)
anomaly_summary = pd.Series({
    "total_flights": len(analytics_df),
    "duration_anomalies": int(analytics_df["duration_anomaly_flag"].sum()),
    "anomaly_rate_pct": round(analytics_df["duration_anomaly_flag"].mean() * 100, 2),
})
anomaly_summary

total_flights         1005.0
duration_anomalies       1.0
anomaly_rate_pct         0.1
dtype: float64

In [23]:
# 4. Distribution of Flights by Airline
airline_distribution = analytics_df["airline"].value_counts().rename("flight_count")
airline_distribution

airline
IndiGo       249
SpiceJet     236
Air India    233
Vistara      218
Unknown       69
Name: flight_count, dtype: int64

## 12. Data Quality Summary

Every number below is pulled from a variable set earlier in the notebook, so it's traceable back
to the exact validation/cleaning step that produced it.

In [24]:
dq["duplicate_flight_id_rows_found"] = int(n_dup_flight_ids)
dq["flight_id_format_issues_found"] = int(n_bad_ids)
dq["referential_integrity_issues_found"] = int(
    n_booking_bad_passenger_ref + n_booking_bad_flight_ref + n_payment_bad_booking_ref
)

summary = pd.Series({
    "-- Raw record counts --": "",
    "Raw flight records": dq["flights_raw_count"],
    "Raw passenger records": dq["passengers_raw_count"],
    "Raw booking records": dq["bookings_raw_count"],
    "Raw payment records": dq["payments_raw_count"],

    "-- Duplicates --": "",
    "Exact duplicate flight rows found": int(n_exact_dup_rows),
    "Exact duplicate flight rows removed": dq["flights_exact_duplicates_removed"],
    "flight_id values reused across >1 row (found, kept - see Sec. 9)": dq["duplicate_flight_id_rows_found"],
    "Exact duplicate rows found (bookings)": dq["bookings_exact_duplicates_found"],
    "Exact duplicate rows found (payments)": dq["payments_exact_duplicates_found"],
    "Exact duplicate rows found (passengers)": dq["passengers_exact_duplicates_found"],

    "-- Missing values (NaN) identified --": "",
    "flights.airline missing": dq["flights_airline_missing_found"],
    "bookings.status missing": dq["bookings_status_missing_found"],
    "passengers.last_name missing": dq["passengers_lastname_missing_found"],
    "payments.amount missing": dq["payments_amount_missing_found"],

    "-- Invalid placeholder values identified --": "",
    "flights.airline = 'UNKNOWN'": dq["flights_airline_unknown_found"],
    "bookings.status = 'INVALID'": dq["bookings_status_invalid_found"],
    "payments.amount = 'INVALID'": dq["payments_amount_invalid_found"],

    "-- Missing / invalid values handled --": "",
    "flights.airline standardized to 'Unknown'": (
        dq["flights_airline_missing_found"] + dq["flights_airline_unknown_found"]
    ),
    "bookings.status standardized to 'Unknown'": (
        dq["bookings_status_missing_found"] + dq["bookings_status_invalid_found"]
    ),
    "passengers.last_name set to 'Unknown'": dq["passengers_lastname_missing_found"],
    "passengers.aadhaar_id zero-padded to 12 digits": dq["passengers_aadhaar_zero_padded"],
    "payments.amount flagged unusable (kept, not dropped)": (
        dq["payments_amount_missing_found"] + dq["payments_amount_invalid_found"]
    ),

    "-- Flagged anomalies (not modified) --": "",
    "Flight duration anomalies": dq["flights_duration_anomalies_found"],
    "Conflicting passenger_id values": dq["passengers_reused_ids_found"],
    "Passenger rows affected by conflicting passenger_id": dq["passengers_reused_id_rows_found"],

    "-- Referential integrity --": "",
    "Referential integrity issues found": dq["referential_integrity_issues_found"],

    "-- Derived --": "",
    "Overnight flights identified": dq["flights_overnight_count"],

    "-- Final output --": "",
    "Final analytics dataset record count": dq["analytics_dataset_row_count"],
})
summary

-- Raw record counts --                                                 
Raw flight records                                                  1020
Raw passenger records                                               1039
Raw booking records                                                 1000
Raw payment records                                                 1000
-- Duplicates --                                                        
Exact duplicate flight rows found                                     15
Exact duplicate flight rows removed                                   15
flight_id values reused across >1 row (found, kept - see Sec. 9)      16
Exact duplicate rows found (bookings)                                  0
Exact duplicate rows found (payments)                                  0
Exact duplicate rows found (passengers)                                0
-- Missing values (NaN) identified --                                   
flights.airline missing                            

## 13. Save Processed Data

Only the final analytics-ready dataset is saved - no unnecessary intermediate files. The raw
workbook in `data/raw/` is left exactly as it was loaded.

In [25]:
output_path = PROCESSED_DIR / "flights_analytics_ready.csv"
analytics_df.to_csv(output_path, index=False)
assert output_path.exists(), "expected output file was not written"
log.info(f"Saved analytics-ready dataset: {output_path} ({len(analytics_df)} rows)")

print("Raw file last modified (unchanged by this pipeline):",
      pd.Timestamp(os.path.getmtime(RAW_PATH), unit="s"))
print("Output file:", output_path.resolve())

INFO: Saved analytics-ready dataset: ..\data\processed\flights_analytics_ready.csv (1005 rows)


Raw file last modified (unchanged by this pipeline): 2026-09-10 12:52:12.527216196
Output file: C:\Users\91915\Downloads\Casestudy\Airlines_Use_Case\data\processed\flights_analytics_ready.csv
